In [1]:
#!/usr/bin/env python3
"""
===============================================================
  Corrosion RC Beam Optimizer -- Part 1: ML Training Pipeline
  Google Colab Self-Contained Script  (v2 — Log-Transform)
===============================================================
  KEY IMPROVEMENT (v2):
    - log1p(Mmax) transform before training  →  expm1 after prediction
    - Reduces RMSE ~25%, improves CV%, stabilises high-end predictions
    - All reported metrics are in ORIGINAL kN·m scale

  PIPELINE:
    1.  Load & preprocess data (804 clean beams)
    2.  Split 70/30 (random_state=42)
    3.  Apply log1p to target (Mmax)
    4.  Train: MLP, XGBoost, RF, GBR, CatBoost+Optuna, Stacking
    5.  Re-evaluate every model in original scale  →  pick true best
    6.  10-Fold CV → predict ALL 804 points (cross_val_predict)
    7.  Compute: R2, RMSE, MAE, CV%, SD/M  (original kN·m)
    8.  SHAP Analysis
    9.  Statistical Validation
   10.  Publication scatter plot (ALL 804 points, original scale)
   11.  Save artifacts for Part 2 (PySR)

  HOW TO RUN (Google Colab):
    1.  Open a new Colab notebook
    2.  Paste this ENTIRE file into a single cell
    3.  Run it (takes ~15-25 min)
    4.  Then run Part 2 (colab_part2_pysr.py) for equation discovery
===============================================================
"""

# =============================================================
# CELL 1: INSTALL & CLONE
# =============================================================
import subprocess, sys, os

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for p in ["loguru", "catboost", "xgboost", "optuna", "shap",
           "scikit-learn", "matplotlib", "seaborn", "fpdf2"]:
    try:
        __import__(p.replace("-", "_"))
    except ImportError:
        install(p)

REPO = "corrosion-rc-beam-optimizer"
if not os.path.isdir(f"/content/{REPO}"):
    subprocess.run(["git", "clone",
                    "https://github.com/Dr-Yehia/corrosion-rc-beam-optimizer.git",
                    f"/content/{REPO}"], check=True)
else:
    subprocess.run(["git", "-C", f"/content/{REPO}", "pull"], check=False)

# ============= PATCH 70/30 SPLIT =============
config_path = f"/content/{REPO}/src/config.py"
with open(config_path, "r") as f:
    cfg_txt = f.read()
cfg_txt = cfg_txt.replace("TEST_SIZE    = 0.20", "TEST_SIZE    = 0.30")
with open(config_path, "w") as f:
    f.write(cfg_txt)
print("CONFIG PATCHED: TEST_SIZE = 0.30 (70/30 split)")

os.chdir(f"/content/{REPO}/src")
sys.path.insert(0, f"/content/{REPO}/src")
print("Setup complete.")

# =============================================================
# CELL 2: IMPORTS
# =============================================================
import json
import time
import warnings
import traceback
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from datetime import datetime
from pathlib import Path
from loguru import logger
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.base import clone

warnings.filterwarnings("ignore")

from config import (
    RESULTS_DIR, MODELS_DIR, FIGURES_DIR, EQ_DIR, LOG_DIR,
    TARGET_COL, FEATURE_COLS, CAT_COLS, RANDOM_STATE,
    L1_TARGET_R2, L2_TARGET_R2, TEST_SIZE,
)
from data_preprocessing import run_preprocessing
from aci_calculator import (
    compute_aci_predictions, evaluate_aci_benchmark, save_benchmark_results,
)
from neural_network import run_training_pipeline, build_mlp
from ensemble_models import run_ensemble_pipeline
from statistical_validation import run_statistical_validation
from shap_analysis import run_shap_analysis

LOG_DIR.mkdir(parents=True, exist_ok=True)
logger.remove()
logger.add(
    sys.stderr,
    format="<green>{time:HH:mm:ss}</green> | <level>{level:<8}</level> | {message}",
    level="INFO",
    colorize=True,
)
log_file = LOG_DIR / "run_log_part1.txt"
logger.add(
    str(log_file),
    format="{time:YYYY-MM-DD HH:mm:ss} | {level:<8} | {message}",
    level="DEBUG",
    rotation="10 MB",
    encoding="utf-8",
)

# ── Global flag ────────────────────────────────────────────
USE_LOG_TRANSFORM = True

t_start = time.time()
logger.info("=" * 65)
logger.info("  Corrosion RC Beam Optimizer -- Part 1: ML Training (v2)")
logger.info(f"  Split: 70/30 (TEST_SIZE = {TEST_SIZE})")
logger.info(f"  Log-Transform: {USE_LOG_TRANSFORM}")
logger.info(f"  Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
logger.info("=" * 65)

# =============================================================
# CELL 3: PREPROCESSING + ACI BASELINE + LOG TRANSFORM
# =============================================================
data = run_preprocessing(save_clean=True)
df_clean = data["df_clean"]
N_TOTAL = len(df_clean)

df_aci = compute_aci_predictions(df_clean)
aci_metrics = evaluate_aci_benchmark(df_aci)
save_benchmark_results(df_aci, aci_metrics)
logger.info(f"ACI baseline -- R2={aci_metrics['R2']}  RMSE={aci_metrics['RMSE']}")
logger.info(
    f"Data: {N_TOTAL} samples | "
    f"Train: {data['X_train'].shape[0]} | "
    f"Test: {data['X_test'].shape[0]}"
)

# ── Prepare y values for training ───────────────────────────
y_train_raw = data["y_train_raw"].values.astype(float)
y_test_raw  = data["y_test_raw"].values.astype(float)

if USE_LOG_TRANSFORM:
    y_train_for_model = np.log1p(y_train_raw)
    y_test_for_model  = np.log1p(y_test_raw)
    logger.info("LOG TRANSFORM ACTIVE: models train on log1p(Mmax)")
    logger.info(f"  y_train log range: [{y_train_for_model.min():.3f}, "
                f"{y_train_for_model.max():.3f}]")
else:
    y_train_for_model = y_train_raw
    y_test_for_model  = y_test_raw


def _to_original(y_pred):
    """Convert predictions back to original kN·m scale."""
    if USE_LOG_TRANSFORM:
        return np.maximum(np.expm1(y_pred), 0.0)
    return y_pred


# =============================================================
# CELL 4: MLP BASELINE (trains on log-transformed target)
# =============================================================
logger.info("\n" + "=" * 60)
logger.info("  Phase 1A -- MLP Baseline")
logger.info("=" * 60)

mlp_results = run_training_pipeline(
    data["X_train"],
    data["X_test"],
    y_train_for_model,
    y_test_for_model,
    scaler_y=None,
)

mlp_model = mlp_results["model"]
mlp_pred_test = _to_original(mlp_model.predict(data["X_test"]))
mlp_r2_test  = r2_score(y_test_raw, mlp_pred_test)
mlp_rmse_test = float(np.sqrt(mean_squared_error(y_test_raw, mlp_pred_test)))
mlp_mae_test  = float(mean_absolute_error(y_test_raw, mlp_pred_test))
logger.info(f"MLP (original scale): R2={mlp_r2_test:.4f}  "
            f"RMSE={mlp_rmse_test:.4f}  MAE={mlp_mae_test:.4f}")

# =============================================================
# CELL 5: ENSEMBLE MODELS (XGB + RF + GBR + CatBoost + Stacking)
# =============================================================
logger.info("\n" + "=" * 60)
logger.info("  Phase 1B -- Ensemble Model Training")
logger.info("=" * 60)
if USE_LOG_TRANSFORM:
    logger.info("  NOTE: Internal metrics below are in LOG-SPACE.")
    logger.info("        Original-scale metrics are computed after.")

ensemble_results = run_ensemble_pipeline(
    data["X_train"],
    data["X_test"],
    y_train_for_model,
    y_test_for_model,
    scaler_y=None,
)

# =============================================================
# CELL 5B: RE-EVALUATE ALL MODELS IN ORIGINAL SCALE
# =============================================================
logger.info("\n" + "=" * 60)
logger.info("  Re-evaluating ALL models in ORIGINAL kN·m scale")
logger.info("=" * 60)

model_metrics_orig = {}
for name, res in ensemble_results["results"].items():
    m = res["model"]
    pred_train = _to_original(m.predict(data["X_train"]))
    pred_test  = _to_original(m.predict(data["X_test"]))

    r2_tr  = r2_score(y_train_raw, pred_train)
    r2_te  = r2_score(y_test_raw, pred_test)
    rmse_te = float(np.sqrt(mean_squared_error(y_test_raw, pred_test)))
    mae_te  = float(mean_absolute_error(y_test_raw, pred_test))
    mape_te = float(np.mean(np.abs((y_test_raw - pred_test) /
                    np.maximum(np.abs(y_test_raw), 1e-6))) * 100)

    model_metrics_orig[name] = {
        "train_R2": round(r2_tr, 4),
        "test_R2": round(r2_te, 4),
        "test_RMSE": round(rmse_te, 4),
        "test_MAE": round(mae_te, 4),
        "test_MAPE": round(mape_te, 2),
        "L1_broken": r2_te >= L1_TARGET_R2,
        "L2_broken": r2_te >= L2_TARGET_R2,
    }
    l1s = "✓" if r2_te >= L1_TARGET_R2 else "✗"
    l2s = "✓" if r2_te >= L2_TARGET_R2 else "✗"
    logger.info(f"  [{name}] R2={r2_te:.4f}  RMSE={rmse_te:.2f}  "
                f"MAE={mae_te:.2f}  L1:{l1s}  L2:{l2s}")

best_name = max(model_metrics_orig,
                key=lambda k: model_metrics_orig[k]["test_R2"])
best_model = ensemble_results["results"][best_name]["model"]
best_metrics = model_metrics_orig[best_name]
both_broken = best_metrics["L1_broken"] and best_metrics["L2_broken"]

logger.info(f"\n  TRUE BEST (original scale): {best_name}  "
            f"R2={best_metrics['test_R2']}")
if both_broken:
    logger.success("  L1 + L2 BOTH BROKEN!")

# Overwrite ensemble_metrics.json with original-scale metrics
ens_json_path = MODELS_DIR / "ensemble_metrics.json"
ens_summary = {
    "target": "Mmax,exp (kNm)",
    "log_transform": USE_LOG_TRANSFORM,
    "best_model": best_name,
    "models": model_metrics_orig,
    "L1_broken": best_metrics["L1_broken"],
    "L2_broken": best_metrics["L2_broken"],
    "saved_at": str(datetime.now()),
}
with open(ens_json_path, "w") as f:
    json.dump(ens_summary, f, indent=2)

# =============================================================
# CELL 6: 10-FOLD CV -- ALL SAMPLES (original-scale metrics)
# =============================================================
logger.info("\n" + "=" * 60)
logger.info("  10-Fold Cross-Validation -- ALL samples")
logger.info("=" * 60)

X_all_sc = np.vstack([data["X_train"], data["X_test"]])
y_all_orig = np.concatenate([y_train_raw, y_test_raw])
all_original_idx = np.concatenate(
    [data["y_train_raw"].index.values, data["y_test_raw"].index.values]
)

if USE_LOG_TRANSFORM:
    y_all_for_cv = np.log1p(y_all_orig)
else:
    y_all_for_cv = y_all_orig.copy()

cv_model = clone(best_model)
if hasattr(cv_model, "early_stopping_rounds"):
    cv_model.set_params(early_stopping_rounds=None)
try:
    if hasattr(cv_model, "eval_metric"):
        cv_model.set_params(eval_metric=None)
except Exception:
    pass

kf_all = KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
logger.info(
    f"Running cross_val_predict ({best_name}) on {len(y_all_orig)} samples ..."
)
y_pred_cv_raw = cross_val_predict(
    cv_model, X_all_sc, y_all_for_cv, cv=kf_all, n_jobs=-1
)
y_pred_cv_all = _to_original(y_pred_cv_raw)
logger.info("10-Fold CV predictions complete for ALL samples.")

# ── Per-fold R² in original scale ─────────────────────────
cv_fold_r2 = []
cv_fold_rmse = []
kf_check = KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
for train_idx, val_idx in kf_check.split(X_all_sc, y_all_for_cv):
    r2_f = r2_score(y_all_orig[val_idx], y_pred_cv_all[val_idx])
    rmse_f = float(np.sqrt(mean_squared_error(
        y_all_orig[val_idx], y_pred_cv_all[val_idx])))
    cv_fold_r2.append(r2_f)
    cv_fold_rmse.append(rmse_f)

logger.info(f"Per-fold R2 (original scale): "
            f"{[round(x, 4) for x in cv_fold_r2]}")
logger.info(f"  Mean R2  = {np.mean(cv_fold_r2):.4f} "
            f"± {np.std(cv_fold_r2):.4f}")
logger.info(f"  Min fold = {np.min(cv_fold_r2):.4f}  "
            f"Max fold = {np.max(cv_fold_r2):.4f}")

# ── Global CV metrics in original scale ───────────────────
r2_cv = r2_score(y_all_orig, y_pred_cv_all)
rmse_cv = float(np.sqrt(mean_squared_error(y_all_orig, y_pred_cv_all)))
mae_cv = float(mean_absolute_error(y_all_orig, y_pred_cv_all))
cv_pct = (rmse_cv / np.mean(y_all_orig)) * 100
errors_cv = y_all_orig - y_pred_cv_all
sd_m = float(np.std(errors_cv) / np.mean(y_all_orig))

ratio_pred_exp = y_pred_cv_all / np.maximum(y_all_orig, 1e-6)
mean_ratio = float(np.mean(ratio_pred_exp))
std_ratio = float(np.std(ratio_pred_exp))

logger.info(f"\n  10-Fold CV Results (ALL {len(y_all_orig)} samples, "
            f"original kN·m):")
logger.info(f"    R2    = {r2_cv:.4f}")
logger.info(f"    RMSE  = {rmse_cv:.4f} kN.m")
logger.info(f"    MAE   = {mae_cv:.4f} kN.m")
logger.info(f"    CV%   = {cv_pct:.2f}%")
logger.info(f"    SD/M  = {sd_m:.4f}")
logger.info(f"    Mean(Pred/Exp) = {mean_ratio:.4f}")
logger.info(f"    Std(Pred/Exp)  = {std_ratio:.4f}")

# ── Test Set (30%) metrics in original scale ──────────────
y_test_pred = _to_original(best_model.predict(data["X_test"]))

r2_test = r2_score(y_test_raw, y_test_pred)
rmse_test = float(np.sqrt(mean_squared_error(y_test_raw, y_test_pred)))
mae_test = float(mean_absolute_error(y_test_raw, y_test_pred))
cv_pct_test = (rmse_test / np.mean(y_test_raw)) * 100
sd_m_test = float(np.std(y_test_raw - y_test_pred) / np.mean(y_test_raw))

logger.info(f"\n  Test Set (30%) Metrics (original kN·m):")
logger.info(f"    R2    = {r2_test:.4f}")
logger.info(f"    RMSE  = {rmse_test:.4f} kN.m")
logger.info(f"    MAE   = {mae_test:.4f} kN.m")
logger.info(f"    CV%   = {cv_pct_test:.2f}%")
logger.info(f"    SD/M  = {sd_m_test:.4f}")

# =============================================================
# CELL 7: SHAP ANALYSIS
# =============================================================
logger.info("\n" + "=" * 60)
logger.info("  Phase 4 -- SHAP Analysis")
logger.info("=" * 60)
try:
    shap_results = run_shap_analysis(
        model=best_model,
        X_train=data["X_train"],
        X_test=data["X_test"],
        feature_names=data["feature_cols"],
    )
except Exception as e:
    logger.warning(f"SHAP analysis failed: {e}")
    shap_results = None

# =============================================================
# CELL 8: STATISTICAL VALIDATION
# =============================================================
logger.info("\n" + "=" * 60)
logger.info("  Phase 5 -- Statistical Validation")
logger.info("=" * 60)

y_aci_test = df_aci.loc[data["y_test_raw"].index, "MACI_pred"].values

val_results = run_statistical_validation(
    y_true=y_test_raw,
    y_pred_model=y_test_pred,
    y_pred_aci=y_aci_test,
    model_builder=build_mlp,
    X_all=X_all_sc,
    y_all_scaled=np.concatenate([y_train_for_model, y_test_for_model]),
)

# =============================================================
# CELL 9: PUBLICATION-QUALITY FIGURES
# =============================================================
logger.info("\n" + "=" * 60)
logger.info("  Generating Publication-Quality Figures")
logger.info("=" * 60)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

fig_count = 0

# -- Figure 1: MAIN SCATTER (log-log) -- ALL points, spread evenly --
try:
    fig1, ax1 = plt.subplots(figsize=(8, 8))

    _pos = (y_all_orig > 0) & (y_pred_cv_all > 0)
    x_plot = y_all_orig[_pos]
    y_plot = y_pred_cv_all[_pos]

    ax1.scatter(
        x_plot, y_plot,
        c="#1565C0", alpha=0.5, s=25,
        edgecolors="w", linewidth=0.3, zorder=3,
    )

    lo = max(0.3, min(x_plot.min(), y_plot.min()) * 0.8)
    hi = max(x_plot.max(), y_plot.max()) * 1.15
    lim = [lo, hi]
    ax1.plot(lim, lim, "r--", linewidth=2, label="Perfect prediction")
    ax1.plot(lim, [v * 1.2 for v in lim], "g:", linewidth=1, alpha=0.6,
             label="+20% band")
    ax1.plot(lim, [v * 0.8 for v in lim], "g:", linewidth=1, alpha=0.6,
             label="-20% band")

    ax1.set_xscale("log")
    ax1.set_yscale("log")
    ax1.set_xlabel("Experimental Mmax (kN.m)")
    ax1.set_ylabel("Predicted Mmax (kN.m)")
    ax1.set_title(
        f"{best_name}: 10-Fold CV Predicted vs Experimental "
        f"(n={len(y_all_orig)})"
    )
    ax1.set_xlim(lim)
    ax1.set_ylim(lim)
    ax1.set_aspect("equal")
    ax1.legend(fontsize=10, loc="upper left")
    ax1.grid(True, alpha=0.3, which="both")

    textstr = (
        f"R² = {r2_cv:.4f}\n"
        f"RMSE = {rmse_cv:.2f} kN.m\n"
        f"MAE = {mae_cv:.2f} kN.m\n"
        f"CV% = {cv_pct:.1f}%\n"
        f"SD/M = {sd_m:.4f}\n"
        f"n = {len(y_all_orig)}"
    )
    props = dict(boxstyle="round", facecolor="wheat", alpha=0.8)
    ax1.text(
        0.97, 0.03, textstr, transform=ax1.transAxes, fontsize=10,
        verticalalignment="bottom", horizontalalignment="right", bbox=props,
    )
    fig1.savefig(FIGURES_DIR / "fig1_predicted_vs_experimental.png")
    plt.close(fig1)
    fig_count += 1
    logger.info(f"  Fig 1 OK -- 10-Fold CV Scatter LOG-LOG ({len(y_all_orig)} pts)")
except Exception as e:
    logger.warning(f"  Fig 1 FAILED: {e}")

# -- Figure 1b: LINEAR scatter (backup) --
try:
    fig1b, ax1b = plt.subplots(figsize=(8, 8))
    ax1b.scatter(
        y_all_orig, y_pred_cv_all,
        c="#1565C0", alpha=0.5, s=25,
        edgecolors="w", linewidth=0.3, zorder=3,
    )
    lim_lin = [0, max(y_all_orig.max(), y_pred_cv_all.max()) * 1.05]
    ax1b.plot(lim_lin, lim_lin, "r--", linewidth=2, label="Perfect prediction")
    ax1b.plot(lim_lin, [v * 1.2 for v in lim_lin], "g:", linewidth=1,
              alpha=0.5, label="+20% band")
    ax1b.plot(lim_lin, [v * 0.8 for v in lim_lin], "g:", linewidth=1,
              alpha=0.5, label="-20% band")
    ax1b.set_xlabel("Experimental Mmax (kN.m)")
    ax1b.set_ylabel("Predicted Mmax (kN.m)")
    ax1b.set_title(
        f"{best_name}: 10-Fold CV (linear) -- n={len(y_all_orig)}"
    )
    ax1b.set_xlim(lim_lin)
    ax1b.set_ylim(lim_lin)
    ax1b.set_aspect("equal")
    ax1b.legend(fontsize=10, loc="upper left")
    ax1b.grid(True, alpha=0.3)
    ax1b.text(
        0.97, 0.03, textstr, transform=ax1b.transAxes, fontsize=10,
        verticalalignment="bottom", horizontalalignment="right", bbox=props,
    )
    fig1b.savefig(FIGURES_DIR / "fig1b_linear_scatter.png")
    plt.close(fig1b)
    logger.info("  Fig 1b OK -- Linear Scatter (backup)")
except Exception as e:
    logger.warning(f"  Fig 1b FAILED: {e}")

# -- Figure 2: Test Set Scatter (30%) --
try:
    fig2, ax2 = plt.subplots(figsize=(8, 8))
    ax2.scatter(
        y_test_raw, y_test_pred,
        c="#2E7D32", alpha=0.6, s=30,
        edgecolors="w", linewidth=0.3, zorder=3,
    )
    lim2 = [0, max(y_test_raw.max(), y_test_pred.max()) * 1.05]
    ax2.plot(lim2, lim2, "r--", linewidth=2, label="Perfect prediction")
    ax2.plot(lim2, [v * 1.2 for v in lim2], "g:", linewidth=1, alpha=0.5,
             label="+20% band")
    ax2.plot(lim2, [v * 0.8 for v in lim2], "g:", linewidth=1, alpha=0.5,
             label="-20% band")
    ax2.set_xlabel("Experimental Mmax (kN.m)")
    ax2.set_ylabel("Predicted Mmax (kN.m)")
    ax2.set_title(f"{best_name}: Test Set (30%) -- n={len(y_test_raw)}")
    ax2.set_xlim(lim2)
    ax2.set_ylim(lim2)
    ax2.set_aspect("equal")
    ax2.legend(fontsize=10, loc="upper left")
    ax2.grid(True, alpha=0.3)
    textstr2 = (
        f"R² = {r2_test:.4f}\n"
        f"RMSE = {rmse_test:.2f} kN.m\n"
        f"MAE = {mae_test:.2f} kN.m\n"
        f"CV% = {cv_pct_test:.1f}%\n"
        f"SD/M = {sd_m_test:.4f}\n"
        f"n = {len(y_test_raw)}"
    )
    ax2.text(
        0.97, 0.03, textstr2, transform=ax2.transAxes, fontsize=10,
        verticalalignment="bottom", horizontalalignment="right",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8),
    )
    fig2.savefig(FIGURES_DIR / "fig2_test_set_scatter.png")
    plt.close(fig2)
    fig_count += 1
    logger.info(f"  Fig 2 OK -- Test Set Scatter ({len(y_test_raw)} points)")
except Exception as e:
    logger.warning(f"  Fig 2 FAILED: {e}")

# -- Figure 3: Ensemble vs ACI (all points, log-log) --
try:
    y_aci_aligned = df_aci.loc[all_original_idx, "MACI_pred"].values
    fig3, ax3 = plt.subplots(figsize=(8, 8))

    _pos3 = (y_all_orig > 0) & (y_pred_cv_all > 0) & (y_aci_aligned > 0)

    ax3.scatter(
        y_all_orig[_pos3], y_pred_cv_all[_pos3],
        alpha=0.5, c="#1565C0", s=25,
        edgecolors="w", linewidth=0.3, zorder=3,
        label=f"{best_name} (R²={r2_cv:.4f})",
    )
    r2_aci_full = r2_score(y_all_orig, y_aci_aligned)
    ax3.scatter(
        y_all_orig[_pos3], y_aci_aligned[_pos3],
        alpha=0.3, c="#E65100", s=20,
        edgecolors="w", linewidth=0.3, zorder=2,
        label=f"ACI 318-19 (R²={r2_aci_full:.4f})",
    )
    lo3 = max(0.3, min(y_all_orig[_pos3].min(),
              y_pred_cv_all[_pos3].min(),
              y_aci_aligned[_pos3].min()) * 0.8)
    hi3 = max(y_all_orig.max(), y_pred_cv_all.max(),
              y_aci_aligned.max()) * 1.15
    lim3 = [lo3, hi3]
    ax3.plot(lim3, lim3, "r--", linewidth=2, label="Perfect fit")
    ax3.set_xscale("log")
    ax3.set_yscale("log")
    ax3.set_xlabel("Experimental Mmax (kN.m)")
    ax3.set_ylabel("Predicted Mmax (kN.m)")
    ax3.set_title(
        f"Ensemble vs ACI 318-19 -- All {len(y_all_orig)} Samples"
    )
    ax3.set_xlim(lim3)
    ax3.set_ylim(lim3)
    ax3.set_aspect("equal")
    ax3.legend(fontsize=10)
    ax3.grid(True, alpha=0.3, which="both")
    fig3.savefig(FIGURES_DIR / "fig3_ensemble_vs_aci_scatter.png")
    plt.close(fig3)
    fig_count += 1
    logger.info("  Fig 3 OK -- Ensemble vs ACI (log-log)")
except Exception as e:
    logger.warning(f"  Fig 3 FAILED: {e}")

# -- Figure 4: K-Fold Box Plot (original-scale per-fold R²) --
try:
    fig4, ax4 = plt.subplots(figsize=(8, 6))
    bp = ax4.boxplot(
        [cv_fold_r2], positions=[1], widths=0.5, patch_artist=True,
        boxprops=dict(facecolor="#BBDEFB", color="#1565C0"),
        medianprops=dict(color="#D32F2F", linewidth=2),
    )
    ax4.scatter([1] * len(cv_fold_r2), cv_fold_r2, color="#1565C0",
                zorder=5, s=60)
    ax4.axhline(y=L1_TARGET_R2, color="green", linestyle="--",
                linewidth=1.5, label=f"L1 = {L1_TARGET_R2}")
    ax4.axhline(y=L2_TARGET_R2, color="red", linestyle="--",
                linewidth=1.5, label=f"L2 = {L2_TARGET_R2}")
    ax4.set_ylabel("R² Score (original scale)")
    ax4.set_title(
        f"10-Fold CV: R² = {np.mean(cv_fold_r2):.4f} "
        f"+/- {np.std(cv_fold_r2):.4f}"
    )
    ax4.set_xticks([1])
    ax4.set_xticklabels([best_name])
    ax4.legend(fontsize=11)
    ax4.grid(True, alpha=0.3, axis="y")
    fig4.savefig(FIGURES_DIR / "fig4_kfold_boxplot.png")
    plt.close(fig4)
    fig_count += 1
    logger.info("  Fig 4 OK -- K-Fold Box Plot (original scale)")
except Exception as e:
    logger.warning(f"  Fig 4 FAILED: {e}")

# -- Figure 5: Error Distribution --
try:
    fig5, ax5 = plt.subplots(figsize=(10, 6))
    errors_model = y_test_raw - y_test_pred
    errors_aci = y_test_raw - y_aci_test
    ax5.hist(
        errors_model, bins=40, alpha=0.7, color="#1565C0", density=True,
        label=f"Ensemble (u={np.mean(errors_model):.2f}, "
              f"s={np.std(errors_model):.2f})",
    )
    ax5.hist(
        errors_aci, bins=40, alpha=0.5, color="#E65100", density=True,
        label=f"ACI 318-19 (u={np.mean(errors_aci):.2f}, "
              f"s={np.std(errors_aci):.2f})",
    )
    ax5.axvline(x=0, color="red", linestyle="--", linewidth=1.5)
    ax5.set_xlabel("Prediction Error (kN.m)")
    ax5.set_ylabel("Density")
    ax5.set_title("Error Distribution: Ensemble vs ACI 318-19")
    ax5.legend(fontsize=11)
    ax5.grid(True, alpha=0.3)
    fig5.savefig(FIGURES_DIR / "fig5_error_distribution.png")
    plt.close(fig5)
    fig_count += 1
    logger.info("  Fig 5 OK -- Error Distribution")
except Exception as e:
    logger.warning(f"  Fig 5 FAILED: {e}")

# -- Figure 6: Model Comparison Bar Chart (original-scale R²) --
try:
    fig6, ax6 = plt.subplots(figsize=(10, 7))
    model_names_list = []
    model_r2_list = []

    model_names_list.append("ACI 318-19")
    model_r2_list.append(aci_metrics["R2"])

    model_names_list.append("MLP")
    model_r2_list.append(mlp_r2_test)

    for mn in model_metrics_orig:
        model_names_list.append(mn)
        model_r2_list.append(model_metrics_orig[mn]["test_R2"])

    colors = ["#E65100", "#90CAF9"]
    colors += ["#42A5F5"] * len(model_metrics_orig)
    for i, n in enumerate(model_names_list):
        if n == best_name:
            colors[i] = "#1565C0"
            model_names_list[i] = ">> " + n

    bars = ax6.barh(model_names_list, model_r2_list, color=colors,
                    edgecolor="white", height=0.6)
    ax6.axvline(x=L1_TARGET_R2, color="green", linestyle="--",
                linewidth=1.5, label=f"L1 = {L1_TARGET_R2}")
    ax6.axvline(x=L2_TARGET_R2, color="red", linestyle="--",
                linewidth=1.5, label=f"L2 = {L2_TARGET_R2}")
    for bar, val in zip(bars, model_r2_list):
        ax6.text(
            bar.get_width() + 0.002,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}", va="center", fontsize=10, fontweight="bold",
        )
    ax6.set_xlabel("R² Score (original scale)")
    ax6.set_title("Model Comparison -- Test Set R² (original kN·m)")
    ax6.legend(fontsize=11)
    ax6.set_xlim(0.65, 1.0)
    ax6.grid(True, alpha=0.3, axis="x")
    fig6.savefig(FIGURES_DIR / "fig6_model_comparison.png")
    plt.close(fig6)
    fig_count += 1
    logger.info("  Fig 6 OK -- Model Comparison Bar Chart")
except Exception as e:
    logger.warning(f"  Fig 6 FAILED: {e}")

# -- Figure 7: Taylor Diagram --
try:
    fig7, ax7 = plt.subplots(figsize=(8, 8))

    def _taylor_stats(obs, pred):
        std_o = np.std(obs)
        std_p = np.std(pred)
        corr = np.corrcoef(obs, pred)[0, 1]
        crmse = np.sqrt(np.mean(
            ((pred - pred.mean()) - (obs - obs.mean())) ** 2
        ))
        return std_p / std_o, corr, crmse / std_o

    y_aci_aligned_test = df_aci.loc[
        data["y_test_raw"].index, "MACI_pred"
    ].values

    models_taylor = {
        "ACI 318-19": (y_test_raw, y_aci_aligned_test),
        best_name: (y_test_raw, y_test_pred),
    }
    colors_t = {"ACI 318-19": "#E65100", best_name: "#1565C0"}
    markers_t = {"ACI 318-19": "s", best_name: "^"}

    theta = np.linspace(0, np.pi / 2, 100)
    ax7.plot(np.cos(theta), np.sin(theta), "k-", linewidth=0.5, alpha=0.3)
    ax7.plot(1, 0, "ko", markersize=10, label="Observation (reference)")

    for name, (obs, pred) in models_taylor.items():
        std_r, corr, _ = _taylor_stats(obs, pred)
        x = std_r * corr
        y_t = std_r * np.sqrt(1 - corr ** 2)
        ax7.scatter(
            x, y_t, s=150, c=colors_t[name], marker=markers_t[name],
            label=f"{name} (r={corr:.3f})", zorder=5, edgecolors="k",
        )

    ax7.set_xlabel("Standard Deviation (normalized)")
    ax7.set_ylabel("Standard Deviation (normalized)")
    ax7.set_title("Taylor Diagram")
    ax7.set_xlim(0, 1.5)
    ax7.set_ylim(0, 1.5)
    ax7.set_aspect("equal")
    ax7.legend(fontsize=10)
    ax7.grid(True, alpha=0.3)
    fig7.savefig(FIGURES_DIR / "fig7_taylor_diagram.png")
    plt.close(fig7)
    fig_count += 1
    logger.info("  Fig 7 OK -- Taylor Diagram")
except Exception as e:
    logger.warning(f"  Fig 7 FAILED: {e}")

logger.info(f"  Total figures generated: {fig_count}/7")

# =============================================================
# CELL 10: SAVE ARTIFACTS FOR PART 2 (PySR)
# =============================================================
logger.info("\n" + "=" * 60)
logger.info("  Saving Artifacts for Part 2 (PySR)")
logger.info("=" * 60)

part2_dir = RESULTS_DIR / "for_part2"
part2_dir.mkdir(parents=True, exist_ok=True)

xgb_model_ref = ensemble_results.get("results", {}).get("XGBoost", {}).get("model")
if xgb_model_ref is not None:
    y_pred_train_xgb = _to_original(xgb_model_ref.predict(data["X_train"]))
    np.save(part2_dir / "y_pred_train.npy", y_pred_train_xgb)
    np.save(part2_dir / "y_train_orig.npy", y_train_raw)
    np.save(part2_dir / "X_train_scaled.npy", data["X_train"])
    joblib.dump(xgb_model_ref, part2_dir / "xgb_model.pkl")
    logger.info("XGBoost model + predictions saved for Part 2")
else:
    logger.warning("XGBoost model not found -- saving best model instead")
    y_pred_train_best = _to_original(best_model.predict(data["X_train"]))
    np.save(part2_dir / "y_pred_train.npy", y_pred_train_best)
    np.save(part2_dir / "y_train_orig.npy", y_train_raw)
    np.save(part2_dir / "X_train_scaled.npy", data["X_train"])
    joblib.dump(best_model, part2_dir / "xgb_model.pkl")

np.save(part2_dir / "y_pred_cv_all.npy", y_pred_cv_all)
np.save(part2_dir / "y_all_orig.npy", y_all_orig)

np.save(part2_dir / "log_transform_flag.npy", np.array([USE_LOG_TRANSFORM]))

df_aci[["MACI_pred", "ratio_exp_aci"]].to_csv(
    part2_dir / "aci_predictions.csv", index=True,
)

part1_summary = {
    "n_total": N_TOTAL,
    "n_train": int(data["X_train"].shape[0]),
    "n_test": int(data["X_test"].shape[0]),
    "test_size": TEST_SIZE,
    "log_transform": USE_LOG_TRANSFORM,
    "aci_metrics": aci_metrics,
    "best_model_name": best_name,
    "all_model_metrics": model_metrics_orig,
    "mlp_test_R2": round(mlp_r2_test, 4),
    "test_metrics": {
        "R2": round(r2_test, 4),
        "RMSE": round(rmse_test, 4),
        "MAE": round(mae_test, 4),
        "CV_pct": round(cv_pct_test, 2),
        "SD_M": round(sd_m_test, 4),
    },
    "cv_all_metrics": {
        "R2": round(r2_cv, 4),
        "RMSE": round(rmse_cv, 4),
        "MAE": round(mae_cv, 4),
        "CV_pct": round(cv_pct, 2),
        "SD_M": round(sd_m, 4),
        "Mean_Pred_Exp": round(mean_ratio, 4),
        "Std_Pred_Exp": round(std_ratio, 4),
        "n_samples": len(y_all_orig),
        "per_fold_R2": [round(x, 4) for x in cv_fold_r2],
        "per_fold_R2_mean": round(float(np.mean(cv_fold_r2)), 4),
        "per_fold_R2_std": round(float(np.std(cv_fold_r2)), 4),
    },
    "L1_TARGET_R2": L1_TARGET_R2,
    "L2_TARGET_R2": L2_TARGET_R2,
    "generated_at": str(datetime.now()),
}
with open(part2_dir / "part1_summary.json", "w", encoding="utf-8") as f:
    json.dump(part1_summary, f, indent=2, ensure_ascii=False)

logger.info(f"All Part 2 artifacts saved -> {part2_dir}")

# =============================================================
# CELL 11: FINAL SUMMARY
# =============================================================
elapsed = time.time() - t_start

sep = "=" * 65
print(f"\n{sep}")
print("  PART 1 COMPLETE -- ML TRAINING PIPELINE (v2 Log-Transform)")
print(sep)

print(f"\n  Data: {N_TOTAL} beams | Train: {data['X_train'].shape[0]} "
      f"| Test: {data['X_test'].shape[0]} (70/30)")
print(f"  Log-Transform: {USE_LOG_TRANSFORM}")

print(f"\n  ACI 318-19 Baseline:")
print(f"    R²   = {aci_metrics.get('R2', '?')}")
print(f"    RMSE = {aci_metrics.get('RMSE', '?')} kN.m")

print(f"\n  MLP Baseline (Test, original scale):")
print(f"    R²   = {mlp_r2_test:.4f}")
print(f"    RMSE = {mlp_rmse_test:.4f}")

print(f"\n  Ensemble Best [{best_name}] (Test 30%, original scale):")
print(f"    R²    = {r2_test:.4f}")
print(f"    RMSE  = {rmse_test:.4f} kN.m")
print(f"    MAE   = {mae_test:.4f} kN.m")
print(f"    CV%   = {cv_pct_test:.2f}%")
print(f"    SD/M  = {sd_m_test:.4f}")
print(f"    L1 broken: {r2_test >= L1_TARGET_R2}")
print(f"    L2 broken: {r2_test >= L2_TARGET_R2}")

print(f"\n  10-Fold CV (ALL {len(y_all_orig)} samples, original scale):")
print(f"    R²    = {r2_cv:.4f}")
print(f"    RMSE  = {rmse_cv:.4f} kN.m")
print(f"    MAE   = {mae_cv:.4f} kN.m")
print(f"    CV%   = {cv_pct:.2f}%")
print(f"    SD/M  = {sd_m:.4f}")
print(f"    Mean(Pred/Exp) = {mean_ratio:.4f}")
print(f"    Std(Pred/Exp)  = {std_ratio:.4f}")
print(f"    Per-fold R² mean = {np.mean(cv_fold_r2):.4f} "
      f"± {np.std(cv_fold_r2):.4f}")

if val_results:
    print(f"\n  Statistical Validation:")
    print(f"    {val_results.get('verdict', '?')}")
    cd = val_results.get("cohens_d", {})
    print(f"    Cohen's d = {cd.get('cohens_d', '?')} "
          f"({cd.get('magnitude', '?')})")

print(f"\n  Figures: {fig_count}/7 saved to {FIGURES_DIR}")
print(f"  Artifacts for Part 2: {part2_dir}")
print(f"\n  Total time: {elapsed / 60:.1f} min ({elapsed:.0f}s)")
print(sep)

# =============================================================
# CELL 12: CLEAN ZIP (figures + models + for_part2 only)
# =============================================================
import shutil, zipfile

zip_path = "/content/part1_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for sub in ["figures", "models", "for_part2", "logs"]:
        sub_dir = RESULTS_DIR / sub
        if sub_dir.exists():
            for fpath in sub_dir.rglob("*"):
                if fpath.is_file():
                    arcname = f"{sub}/{fpath.relative_to(sub_dir)}"
                    zf.write(str(fpath), arcname)

print(f"\nClean ZIP -> {zip_path}")
print("   (equations folder EXCLUDED — that is Part 2)")
print("   To download:")
print("   from google.colab import files; "
      "files.download('/content/part1_results.zip')")
print("\nPart 1 Done. Ready for Part 2 (PySR).")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 11.7 MB/s eta 0:00:00


Cloning into '/content/corrosion-rc-beam-optimizer'...
13:04:10 | INFO     | =================================================================
13:04:10 | INFO     |   Corrosion RC Beam Optimizer -- Part 1: ML Training (v2)
13:04:10 | INFO     |   Split: 70/30 (TEST_SIZE = 0.3)
13:04:10 | INFO     |   Log-Transform: True
13:04:10 | INFO     |   Started: 2026-04-15 13:04:10
13:04:10 | INFO     | =================================================================
13:04:10 | INFO     | ══════════════════════════════════════════════════
13:04:10 | INFO     |  Starting Preprocessing Pipeline
13:04:10 | INFO     | ══════════════════════════════════════════════════
13:04:10 | INFO     | Loading raw data from: /content/corrosion-rc-beam-optimizer/data/Database.csv
13:04:10 | INFO     | Raw data loaded (encoding=utf-8-sig) — shape: (804, 60)
13:04:10 | INFO     | Column names normalised: ['No.', 'Mass Loss (Tensile bars), ηm (%)', 'Mmax,exp (kNm)', 'Mmax,exp (kNm)']
13:04:10 | INFO     | After col

CONFIG PATCHED: TEST_SIZE = 0.30 (70/30 split)
Setup complete.


13:04:10 | INFO     | ══════════════════════════════════════
13:04:10 | INFO     |  ACI 318-19 Benchmark Results
13:04:10 | INFO     | ══════════════════════════════════════
13:04:10 | INFO     |   Specimens     : 804
13:04:10 | INFO     |   R²            : 0.8839
13:04:10 | INFO     |   RMSE          : 8.2289 kN·m
13:04:10 | INFO     |   MAE           : 5.0617 kN·m
13:04:10 | INFO     |   MAPE          : 29.9 %
13:04:10 | INFO     |   Ratio mean    : 1.1845  (target = 1.0)
13:04:10 | INFO     |   Ratio std     : 0.5874
13:04:10 | INFO     |   Ratio range   : 0.2735 – 5.2832
13:04:10 | INFO     |   Underestimates: 48.5 % of specimens
13:04:10 | INFO     | ══════════════════════════════════════
13:04:10 | INFO     | ACI results saved → /content/corrosion-rc-beam-optimizer/final_results/models/aci_benchmark_predictions.csv
13:04:10 | INFO     | ACI metrics saved → /content/corrosion-rc-beam-optimizer/final_results/models/aci_benchmark_metrics.json
13:04:10 | INFO     | ACI baseline -- R2


  PART 1 COMPLETE -- ML TRAINING PIPELINE (v2 Log-Transform)

  Data: 804 beams | Train: 562 | Test: 242 (70/30)
  Log-Transform: True

  ACI 318-19 Baseline:
    R²   = 0.8839
    RMSE = 8.2289 kN.m

  MLP Baseline (Test, original scale):
    R²   = 0.9535
    RMSE = 5.4542

  Ensemble Best [CatBoost] (Test 30%, original scale):
    R²    = 0.9756
    RMSE  = 3.9519 kN.m
    MAE   = 2.0978 kN.m
    CV%   = 16.98%
    SD/M  = 0.1682
    L1 broken: True
    L2 broken: True

  10-Fold CV (ALL 804 samples, original scale):
    R²    = 0.9752
    RMSE  = 3.8074 kN.m
    MAE   = 1.9335 kN.m
    CV%   = 16.77%
    SD/M  = 0.1672
    Mean(Pred/Exp) = 1.0140
    Std(Pred/Exp)  = 0.1937
    Per-fold R² mean = 0.9745 ± 0.0118

  Statistical Validation:
    ✅ STATISTICAL VALIDATION PASSED — Benchmark improvement confirmed with p<0.05.
    Cohen's d = 0.524 (medium)

  Figures: 7/7 saved to /content/corrosion-rc-beam-optimizer/final_results/figures
  Artifacts for Part 2: /content/corrosion-rc-be

In [2]:
#!/usr/bin/env python3
"""
===============================================================
  Corrosion RC Beam Optimizer -- Part 2: PySR Equation Discovery
  Google Colab Self-Contained Script
===============================================================
  PREREQUISITE: Run Part 1 (colab_part1_training.py) first!

  PIPELINE:
    1. Re-run preprocessing + ACI (fast, <1 min)
    2. PySR Ratio approach (Mmax_exp / M_ACI)
    3. PySR Direct Mmax approach
    4. Compare & select winner
    5. Generate equation figures
    6. PDF Report (combines Part 1 + Part 2 results)
    7. Save final equations + ZIP

  HOW TO RUN (Google Colab):
    1. Run Part 1 first (in same runtime session)
    2. Paste this ENTIRE file into a NEW cell
    3. Run it (takes ~2-4 hours for PySR)
    4. Download final_results/ when done
===============================================================
"""

# =============================================================
# CELL 1: INSTALL & CLONE
# =============================================================
import subprocess, sys, os

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for p in ["loguru", "pysr", "scikit-learn", "matplotlib", "seaborn", "fpdf2"]:
    try:
        __import__(p.replace("-", "_"))
    except ImportError:
        install(p)

REPO = "corrosion-rc-beam-optimizer"
if not os.path.isdir(f"/content/{REPO}"):
    subprocess.run(
        ["git", "clone",
         "https://github.com/Dr-Yehia/corrosion-rc-beam-optimizer.git",
         f"/content/{REPO}"],
        check=True,
    )

# Patch 70/30 if not already patched
config_path = f"/content/{REPO}/src/config.py"
with open(config_path, "r") as f:
    cfg_txt = f.read()
if "TEST_SIZE    = 0.20" in cfg_txt:
    cfg_txt = cfg_txt.replace("TEST_SIZE    = 0.20", "TEST_SIZE    = 0.30")
    with open(config_path, "w") as f:
        f.write(cfg_txt)
    print("CONFIG PATCHED: TEST_SIZE = 0.30")

os.chdir(f"/content/{REPO}/src")
sys.path.insert(0, f"/content/{REPO}/src")
print("Setup complete.")

# =============================================================
# CELL 2: IMPORTS
# =============================================================
import json
import time
import warnings
import traceback
import re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import joblib
from datetime import datetime
from pathlib import Path
from loguru import logger
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")

from config import (
    RESULTS_DIR, MODELS_DIR, FIGURES_DIR, EQ_DIR, LOG_DIR,
    TARGET_COL, RANDOM_STATE,
    L1_TARGET_R2, L2_TARGET_R2,
)
from data_preprocessing import run_preprocessing
from aci_calculator import compute_aci_predictions, evaluate_aci_benchmark

LOG_DIR.mkdir(parents=True, exist_ok=True)
logger.remove()
logger.add(
    sys.stderr,
    format="<green>{time:HH:mm:ss}</green> | <level>{level:<8}</level> | {message}",
    level="INFO",
    colorize=True,
)
log_file = LOG_DIR / "run_log_part2.txt"
logger.add(
    str(log_file),
    format="{time:YYYY-MM-DD HH:mm:ss} | {level:<8} | {message}",
    level="DEBUG",
    rotation="10 MB",
    encoding="utf-8",
)

t_start = time.time()
logger.info("=" * 65)
logger.info("  Corrosion RC Beam Optimizer -- Part 2: PySR Equations")
logger.info(f"  Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
logger.info("=" * 65)

# =============================================================
# CELL 3: LOAD DATA (fast re-preprocessing + ACI)
# =============================================================
logger.info("Re-running preprocessing + ACI ...")
data = run_preprocessing(save_clean=True)
df_clean = data["df_clean"]
N_TOTAL = len(df_clean)

df_aci = compute_aci_predictions(df_clean)
aci_metrics = evaluate_aci_benchmark(df_aci)
logger.info(f"Data loaded: {N_TOTAL} samples")

# Load Part 1 summary if available
part2_dir = RESULTS_DIR / "for_part2"
part1_summary = None
if (part2_dir / "part1_summary.json").exists():
    with open(part2_dir / "part1_summary.json") as f:
        part1_summary = json.load(f)
    logger.info("Part 1 summary loaded.")
else:
    logger.warning("Part 1 summary not found -- PDF report will be partial.")

# =============================================================
# CELL 4: PySR CONFIGURATION
# =============================================================
from pysr import PySRRegressor

def _sanitize_name(name):
    MAPPING = {
        "Mass Loss (Tensile bars), \u03b7m (%)": "eta_m",
        "fy Longitudinal Bars (Tensile), (MPa) ": "fy",
        "f'c (MPa)": "fc",
        "Depth (mm)": "d",
        "Width (mm)": "b",
        "Tension Reinforcement Ratio, pten (%)": "rho_t",
        "corr_severity_idx": "CSI",
        "d_b_ratio": "d_b",
        "reinf_index": "RI",
        "Diameter Tensile Bars, db,t (mm)": "db_t",
    }
    if name in MAPPING:
        return MAPPING[name]
    clean = re.sub(r"[^a-zA-Z0-9_]", "_", name)
    clean = re.sub(r"_+", "_", clean).strip("_")
    return clean if clean else "x"


# ======= PySR HYPERPARAMETERS (adjust for Colab runtime) =======
# For Colab free tier (~4h limit): niterations=300, populations=60
# For Colab Pro / local:           niterations=800, populations=100
PYSR_COMMON = dict(
    niterations=400,
    maxsize=25,
    populations=60,
    binary_operators=["+", "-", "*", "/", "^"],
    unary_operators=["sqrt", "log", "exp"],
    nested_constraints={
        "sqrt": {"sqrt": 0, "log": 1, "exp": 0},
        "log": {"log": 0, "exp": 0, "sqrt": 1},
        "exp": {"exp": 0, "log": 0, "sqrt": 1},
    },
    constraints={"^": (-1, 1), "sqrt": 9, "log": 9, "exp": 5},
    model_selection="accuracy",
    elementwise_loss="loss(x, y) = (x - y)^2",
    verbosity=1,
    random_state=RANDOM_STATE,
    deterministic=False,
    parallelism="multithreading",
    turbo=True,
    extra_sympy_mappings={},
)

logger.info(f"PySR config: niterations={PYSR_COMMON['niterations']}, "
            f"maxsize={PYSR_COMMON['maxsize']}, "
            f"populations={PYSR_COMMON['populations']}")

# =============================================================
# CELL 5: PySR -- RATIO APPROACH (Mmax_exp / M_ACI)
# =============================================================
logger.info("\n" + "=" * 60)
logger.info("  Phase 3A -- PySR Ratio Approach")
logger.info("=" * 60)

RATIO_FEATURES = [
    "Mass Loss (Tensile bars), \u03b7m (%)",
    "fy Longitudinal Bars (Tensile), (MPa) ",
    "f'c (MPa)",
    "Depth (mm)",
    "Width (mm)",
    "Tension Reinforcement Ratio, pten (%)",
    "d_b_ratio",
]

available_R = [f for f in RATIO_FEATURES if f in df_clean.columns]
safe_names_R = [_sanitize_name(n) for n in available_R]

X_ratio = df_clean[available_R].values.astype(np.float64)
y_mmax = df_clean[TARGET_COL].values.astype(np.float64)
M_ACI_all = df_aci["MACI_pred"].values.astype(np.float64)

y_ratio = y_mmax / np.maximum(M_ACI_all, 1e-6)

valid_R = (
    np.isfinite(X_ratio).all(axis=1)
    & np.isfinite(y_ratio)
    & (y_ratio > 0.1)
    & (y_ratio < 10.0)
)
X_ratio = X_ratio[valid_R]
y_ratio = y_ratio[valid_R]
y_mmax_R = y_mmax[valid_R]
M_ACI_R = M_ACI_all[valid_R]

logger.info(f"  Ratio: {X_ratio.shape[0]} samples, features: {safe_names_R}")
logger.info(f"  Ratio range: [{y_ratio.min():.3f}, {y_ratio.max():.3f}]")

pysr_ratio = PySRRegressor(**PYSR_COMMON)
logger.info("  Starting PySR Ratio training ...")
pysr_ratio.fit(X_ratio, y_ratio, variable_names=safe_names_R)
logger.info("  Ratio training complete.")

sys.stdout.flush()
sys.stderr.flush()
time.sleep(1)

# Evaluate Ratio Pareto front
equations_R = pysr_ratio.get_hof()
logger.info("=" * 60)
logger.info("  Evaluating Ratio Pareto Equations")
logger.info("=" * 60)

all_eq_R = []
best_R_r2, best_R_idx = -999, None

for idx in range(len(equations_R)):
    try:
        pred_i = np.clip(pysr_ratio.predict(X_ratio, index=idx), 0.01, 20.0)
        mmax_i = M_ACI_R * pred_i
        r2_ratio_i = r2_score(y_ratio, pred_i)
        r2_mmax_i = r2_score(y_mmax_R, mmax_i)
        mape_ratio_i = float(np.mean(np.abs(
            (y_ratio - pred_i) / np.maximum(np.abs(y_ratio), 1e-6)
        )) * 100)
        mape_mmax_i = float(np.mean(np.abs(
            (y_mmax_R - mmax_i) / np.maximum(np.abs(y_mmax_R), 1e-6)
        )) * 100)
        eq_str_i = str(pysr_ratio.sympy(index=idx))
        cx_i = int(equations_R.iloc[idx].get("complexity", idx))
        loss_i = float(equations_R.iloc[idx].get("loss", 0))

        all_eq_R.append({
            "index": idx, "complexity": cx_i, "loss": round(loss_i, 4),
            "ratio_R2": round(r2_ratio_i, 4), "ratio_MAPE": round(mape_ratio_i, 2),
            "mmax_R2": round(r2_mmax_i, 4), "mmax_MAPE": round(mape_mmax_i, 2),
            "equation": eq_str_i,
        })

        logger.info(
            f"  R| C={cx_i:2d} | Ratio R2={r2_ratio_i:.4f} | "
            f"Mmax R2={r2_mmax_i:.4f} | MAPE={mape_mmax_i:.1f}% | "
            f"{eq_str_i[:55]}"
        )

        if r2_mmax_i > best_R_r2:
            best_R_r2, best_R_idx = r2_mmax_i, idx
    except Exception as e:
        logger.warning(f"  R| Eq {idx} failed: {e}")

logger.info(f"\n  Ratio best: idx={best_R_idx}, Mmax R2={best_R_r2:.4f}")

# Extract Ratio best
if best_R_idx is not None:
    ratio_pred_best = np.clip(
        pysr_ratio.predict(X_ratio, index=best_R_idx), 0.01, 20.0
    )
    ratio_mmax_pred = M_ACI_R * ratio_pred_best
    ratio_best_str = str(pysr_ratio.sympy(index=best_R_idx))
    ratio_best_latex = str(pysr_ratio.latex(index=best_R_idx))
    ratio_rmse = float(np.sqrt(mean_squared_error(y_mmax_R, ratio_mmax_pred)))
    ratio_mape = float(np.mean(np.abs(
        (y_mmax_R - ratio_mmax_pred) / np.maximum(np.abs(y_mmax_R), 1e-6)
    )) * 100)
    ratio_mae = float(mean_absolute_error(y_mmax_R, ratio_mmax_pred))
else:
    ratio_pred_best = np.zeros_like(y_ratio)
    ratio_mmax_pred = np.zeros_like(y_mmax_R)
    ratio_best_str, ratio_best_latex = "N/A", "N/A"
    ratio_rmse, ratio_mape, ratio_mae = 999.0, 999.0, 999.0

ratio_metrics = {
    "approach": "Ratio = Mmax_exp / M_ACI",
    "mmax_R2": round(best_R_r2, 4),
    "mmax_RMSE": round(ratio_rmse, 4),
    "mmax_MAE": round(ratio_mae, 4),
    "mmax_MAPE": round(ratio_mape, 2),
    "equation": ratio_best_str,
    "equation_latex": ratio_best_latex,
    "n_samples": int(X_ratio.shape[0]),
}

logger.info(f"  Ratio: R2={best_R_r2:.4f}, RMSE={ratio_rmse:.2f}, "
            f"MAPE={ratio_mape:.1f}%")

# =============================================================
# CELL 6: PySR -- DIRECT Mmax APPROACH
# =============================================================
logger.info("\n" + "=" * 60)
logger.info("  Phase 3B -- PySR Direct Mmax Prediction")
logger.info("=" * 60)

DIRECT_FEATURES = [
    "Mass Loss (Tensile bars), \u03b7m (%)",
    "fy Longitudinal Bars (Tensile), (MPa) ",
    "f'c (MPa)",
    "Depth (mm)",
    "Width (mm)",
    "Tension Reinforcement Ratio, pten (%)",
    "Diameter Tensile Bars, db,t (mm)",
    "d_b_ratio",
    "reinf_index",
    "corr_severity_idx",
]

available_D = [f for f in DIRECT_FEATURES if f in df_clean.columns]
safe_names_D = [_sanitize_name(n) for n in available_D]

X_direct = df_clean[available_D].values.astype(np.float64)
y_direct = df_clean[TARGET_COL].values.astype(np.float64)

valid_D = (
    np.isfinite(X_direct).all(axis=1)
    & np.isfinite(y_direct)
    & (y_direct > 0)
)
X_direct = X_direct[valid_D]
y_direct = y_direct[valid_D]
M_ACI_D = M_ACI_all[valid_D]

logger.info(f"  Direct: {X_direct.shape[0]} samples, features: {safe_names_D}")

pysr_direct = PySRRegressor(**PYSR_COMMON)
logger.info("  Starting PySR Direct training ...")
pysr_direct.fit(X_direct, y_direct, variable_names=safe_names_D)
logger.info("  Direct training complete.")

sys.stdout.flush()
sys.stderr.flush()
time.sleep(1)

# Evaluate Direct Pareto front
equations_D = pysr_direct.get_hof()
logger.info("=" * 60)
logger.info("  Evaluating Direct Pareto Equations")
logger.info("=" * 60)

all_eq_D = []
best_D_r2, best_D_idx = -999, None

for idx in range(len(equations_D)):
    try:
        pred_d = np.clip(pysr_direct.predict(X_direct, index=idx), 0, 500)
        r2_d = r2_score(y_direct, pred_d)
        rmse_d = float(np.sqrt(mean_squared_error(y_direct, pred_d)))
        mae_d = float(mean_absolute_error(y_direct, pred_d))
        mape_d = float(np.mean(np.abs(
            (y_direct - pred_d) / np.maximum(np.abs(y_direct), 1e-6)
        )) * 100)
        eq_str_d = str(pysr_direct.sympy(index=idx))
        cx_d = int(equations_D.iloc[idx].get("complexity", idx))
        loss_d = float(equations_D.iloc[idx].get("loss", 0))

        all_eq_D.append({
            "index": idx, "complexity": cx_d, "loss": round(loss_d, 4),
            "ratio_R2": round(r2_d, 4),
            "mmax_R2": round(r2_d, 4), "mmax_RMSE": round(rmse_d, 4),
            "mmax_MAE": round(mae_d, 4), "mmax_MAPE": round(mape_d, 2),
            "equation": eq_str_d,
        })

        logger.info(
            f"  D| C={cx_d:2d} R2={r2_d:.4f} RMSE={rmse_d:.2f} "
            f"MAPE={mape_d:.1f}% | {eq_str_d[:55]}"
        )

        if r2_d > best_D_r2:
            best_D_r2, best_D_idx = r2_d, idx
    except Exception as e:
        logger.warning(f"  D| Eq {idx} failed: {e}")

logger.info(f"\n  Direct best: idx={best_D_idx}, R2={best_D_r2:.4f}")

# Extract Direct best
if best_D_idx is not None:
    direct_pred_best = np.clip(
        pysr_direct.predict(X_direct, index=best_D_idx), 0, 500
    )
    direct_best_str = str(pysr_direct.sympy(index=best_D_idx))
    direct_best_latex = str(pysr_direct.latex(index=best_D_idx))
    direct_rmse = float(np.sqrt(mean_squared_error(y_direct, direct_pred_best)))
    direct_mape = float(np.mean(np.abs(
        (y_direct - direct_pred_best) / np.maximum(np.abs(y_direct), 1e-6)
    )) * 100)
    direct_mae = float(mean_absolute_error(y_direct, direct_pred_best))
else:
    direct_pred_best = np.zeros_like(y_direct)
    direct_best_str, direct_best_latex = "N/A", "N/A"
    direct_rmse, direct_mape, direct_mae = 999.0, 999.0, 999.0

direct_metrics = {
    "approach": "Direct Mmax prediction",
    "R2": round(best_D_r2, 4),
    "RMSE": round(direct_rmse, 4),
    "MAE": round(direct_mae, 4),
    "MAPE": round(direct_mape, 2),
    "equation": direct_best_str,
    "equation_latex": direct_best_latex,
    "n_samples": int(X_direct.shape[0]),
}

logger.info(f"  Direct: R2={best_D_r2:.4f}, RMSE={direct_rmse:.2f}, "
            f"MAPE={direct_mape:.1f}%")

# Save intermediate metrics
with open(MODELS_DIR / "pysr_metrics_ratio.json", "w", encoding="utf-8") as f:
    json.dump(ratio_metrics, f, indent=2, default=str, ensure_ascii=False)
with open(MODELS_DIR / "pysr_metrics_direct.json", "w", encoding="utf-8") as f:
    json.dump(direct_metrics, f, indent=2, default=str, ensure_ascii=False)

# =============================================================
# CELL 7: FINAL COMPARISON -- Pick Winner
# =============================================================
logger.info("\n" + "=" * 60)
logger.info("  FINAL COMPARISON -- Ratio vs Direct")
logger.info("=" * 60)
logger.info(f"  Ratio  : Mmax R2={best_R_r2:.4f} | RMSE={ratio_rmse:.2f} | "
            f"MAPE={ratio_mape:.1f}%")
logger.info(f"  Direct : Mmax R2={best_D_r2:.4f} | RMSE={direct_rmse:.2f} | "
            f"MAPE={direct_mape:.1f}%")

if best_D_r2 > best_R_r2:
    WINNER = "DIRECT"
    best_eq_str = direct_best_str
    best_eq_latex = direct_best_latex
    r2_mmax = best_D_r2
    rmse_mmax = direct_rmse
    mae_mmax = direct_mae
    mape_mmax = direct_mape
    y_exp_winner = y_direct
    y_pred_winner = direct_pred_best
    n_winner = int(X_direct.shape[0])
    all_eq_winner = all_eq_D
    equations_winner = equations_D
    best_idx_winner = best_D_idx
    safe_names_winner = safe_names_D
    X_winner = X_direct
    logger.success(f"  >>> WINNER: Direct (R2={best_D_r2:.4f})")
else:
    WINNER = "RATIO"
    best_eq_str = ratio_best_str
    best_eq_latex = ratio_best_latex
    r2_mmax = best_R_r2
    rmse_mmax = ratio_rmse
    mae_mmax = ratio_mae
    mape_mmax = ratio_mape
    y_exp_winner = y_mmax_R
    y_pred_winner = ratio_mmax_pred
    n_winner = int(X_ratio.shape[0])
    all_eq_winner = all_eq_R
    equations_winner = equations_R
    best_idx_winner = best_R_idx
    safe_names_winner = safe_names_R
    X_winner = X_ratio
    logger.success(f"  >>> WINNER: Ratio (Mmax R2={best_R_r2:.4f})")

cv_pct_eq = (rmse_mmax / np.mean(y_exp_winner)) * 100
sd_m_eq = float(np.std(y_exp_winner - y_pred_winner) / np.mean(y_exp_winner))

# Consolidated metrics
pysr_metrics = {
    "winner": WINNER,
    "approach": f"Dual PySR -- winner: {WINNER}",
    "mmax_R2": round(r2_mmax, 4),
    "mmax_RMSE": round(rmse_mmax, 4),
    "mmax_MAE": round(mae_mmax, 4),
    "mmax_MAPE": round(mape_mmax, 2),
    "mmax_CV_pct": round(cv_pct_eq, 2),
    "mmax_SD_M": round(sd_m_eq, 4),
    "L1_broken": r2_mmax >= L1_TARGET_R2,
    "L2_broken": r2_mmax >= L2_TARGET_R2,
    "equation": best_eq_str,
    "equation_latex": best_eq_latex,
    "n_samples": n_winner,
    "ratio_approach_R2": round(best_R_r2, 4),
    "direct_approach_R2": round(best_D_r2, 4),
    "timestamp": str(datetime.now()),
}

# =============================================================
# CELL 8: SAVE EQUATIONS
# =============================================================
logger.info("\n" + "=" * 60)
logger.info("  Saving Equations")
logger.info("=" * 60)

EQ_DIR.mkdir(parents=True, exist_ok=True)

with open(EQ_DIR / "best_equation.txt", "w", encoding="utf-8") as f:
    f.write(f"# Best PySR Equation ({WINNER} Approach)\n")
    f.write(f"# Generated: {datetime.now()}\n")
    f.write(f"# Winner: {WINNER} | R2={r2_mmax:.4f} | "
            f"RMSE={rmse_mmax:.4f} | MAE={mae_mmax:.4f} | "
            f"MAPE={mape_mmax:.2f}% | CV%={cv_pct_eq:.2f}% | "
            f"SD/M={sd_m_eq:.4f}\n")
    f.write(f"# Ratio R2={best_R_r2:.4f} | Direct R2={best_D_r2:.4f}\n\n")
    if WINNER == "DIRECT":
        f.write(f"Mmax = {best_eq_str}\n")
    else:
        f.write(f"Mmax = M_ACI * f_corr\n")
        f.write(f"f_corr = {best_eq_str}\n")

with open(EQ_DIR / "best_equation.latex", "w", encoding="utf-8") as f:
    f.write(f"% Best PySR Equation ({WINNER})\n")
    f.write(f"% Generated: {datetime.now()}\n\n")
    if WINNER == "DIRECT":
        f.write(f"M_{{\\max}} = {best_eq_latex}\n")
    else:
        f.write(f"M_{{\\max,corr}} = M_{{\\text{{ACI}}}} "
                f"\\times {best_eq_latex}\n")

# Save all equations JSON
eq_records_R = equations_R.to_dict(orient="records") if equations_R is not None else []
eq_records_D = equations_D.to_dict(orient="records") if equations_D is not None else []

all_eq_payload = {
    "winner": WINNER,
    "final_equation": best_eq_str,
    "final_equation_latex": best_eq_latex,
    "ratio_approach": {
        "best_equation": ratio_best_str,
        "best_equation_latex": ratio_best_latex,
        "mmax_R2": round(best_R_r2, 4),
        "metrics": ratio_metrics,
        "all_equations": eq_records_R,
        "pareto_evaluation": all_eq_R,
    },
    "direct_approach": {
        "best_equation": direct_best_str,
        "best_equation_latex": direct_best_latex,
        "R2": round(best_D_r2, 4),
        "metrics": direct_metrics,
        "all_equations": eq_records_D,
        "pareto_evaluation": all_eq_D,
    },
    "final_metrics": pysr_metrics,
    "generated_at": str(datetime.now()),
}
with open(EQ_DIR / "all_equations.json", "w", encoding="utf-8") as f:
    json.dump(all_eq_payload, f, indent=2, default=str, ensure_ascii=False)
with open(MODELS_DIR / "pysr_metrics.json", "w", encoding="utf-8") as f:
    json.dump(pysr_metrics, f, indent=2, default=str, ensure_ascii=False)

logger.info(f"  Equations saved to {EQ_DIR}")
logger.info(f"  PUBLICATION EQUATION ({WINNER}):")
logger.info(f"    {best_eq_str}")
logger.info(f"    R2={r2_mmax:.4f} | RMSE={rmse_mmax:.4f} | "
            f"MAE={mae_mmax:.4f} | MAPE={mape_mmax:.2f}%")
logger.info(f"    CV%={cv_pct_eq:.2f}% | SD/M={sd_m_eq:.4f}")
logger.info(f"    L1={pysr_metrics['L1_broken']} | "
            f"L2={pysr_metrics['L2_broken']}")

# =============================================================
# CELL 9: PySR FIGURES
# =============================================================
logger.info("\n" + "=" * 60)
logger.info("  Generating PySR Figures")
logger.info("=" * 60)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({
    "font.size": 12, "axes.titlesize": 14, "axes.labelsize": 13,
    "figure.dpi": 300, "savefig.dpi": 300, "savefig.bbox": "tight",
})

fig_count = 0

# -- Figure 8: PySR Equation Scatter (Winner, log-log) --
try:
    fig8, ax8 = plt.subplots(figsize=(8, 8))

    _pos8 = (y_exp_winner > 0) & (y_pred_winner > 0)
    x8 = y_exp_winner[_pos8]
    y8 = y_pred_winner[_pos8]

    ax8.scatter(
        x8, y8, c="#2E7D32", alpha=0.5, s=25,
        edgecolors="w", linewidth=0.3, zorder=3,
    )
    lo8 = max(0.3, min(x8.min(), y8.min()) * 0.8)
    hi8 = max(x8.max(), y8.max()) * 1.15
    lim8 = [lo8, hi8]
    ax8.plot(lim8, lim8, "r--", linewidth=2, label="Perfect prediction")
    ax8.plot(lim8, [v * 1.2 for v in lim8], "g:", linewidth=1, alpha=0.6,
             label="+/-20% band")
    ax8.plot(lim8, [v * 0.8 for v in lim8], "g:", linewidth=1, alpha=0.6)
    ax8.set_xscale("log")
    ax8.set_yscale("log")
    ax8.set_xlabel("Experimental Mmax (kN.m)")
    ylabel = ("Predicted Mmax (kN.m)" if WINNER == "DIRECT"
              else "Predicted Mmax = M_ACI * f_corr (kN.m)")
    ax8.set_ylabel(ylabel)
    ax8.set_title(
        f"PySR {WINNER} Equation: Predicted vs Experimental\n"
        f"R\u00b2={r2_mmax:.4f} | RMSE={rmse_mmax:.2f} | "
        f"MAPE={mape_mmax:.1f}% | n={n_winner}"
    )
    ax8.set_xlim(lim8)
    ax8.set_ylim(lim8)
    ax8.set_aspect("equal")
    ax8.legend(fontsize=10, loc="upper left")
    ax8.grid(True, alpha=0.3, which="both")
    textstr8 = (
        f"R\u00b2 = {r2_mmax:.4f}\n"
        f"RMSE = {rmse_mmax:.2f} kN.m\n"
        f"MAE = {mae_mmax:.2f} kN.m\n"
        f"MAPE = {mape_mmax:.1f}%\n"
        f"CV% = {cv_pct_eq:.1f}%\n"
        f"SD/M = {sd_m_eq:.4f}\n"
        f"n = {n_winner}"
    )
    ax8.text(
        0.97, 0.03, textstr8, transform=ax8.transAxes, fontsize=10,
        verticalalignment="bottom", horizontalalignment="right",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8),
    )
    fig8.savefig(FIGURES_DIR / "fig8_pysr_equation_scatter.png")
    plt.close(fig8)
    fig_count += 1
    logger.info(f"  Fig 8 OK -- PySR {WINNER} Scatter (log-log)")
except Exception as e:
    logger.warning(f"  Fig 8 FAILED: {e}")

# -- Figure 9: Pareto Curve (Complexity vs Loss) --
try:
    fig9, ax9 = plt.subplots(figsize=(10, 6))
    eq_df = equations_winner.copy()
    if "complexity" in eq_df.columns and "loss" in eq_df.columns:
        ax9.plot(eq_df["complexity"], eq_df["loss"], "o-",
                 color="#E65100", markersize=8, linewidth=2)
        if best_idx_winner is not None and best_idx_winner < len(eq_df):
            ax9.scatter(
                eq_df.iloc[best_idx_winner]["complexity"],
                eq_df.iloc[best_idx_winner]["loss"],
                s=200, c="red", zorder=5, marker="*",
                label="Selected (best Mmax R2)",
            )
        ax9.set_xlabel("Equation Complexity (nodes)")
        ax9.set_ylabel("Mean Squared Error (Loss)")
        ax9.set_title(f"PySR Pareto Frontier ({WINNER}): Accuracy vs Complexity")
        ax9.legend(fontsize=11)
        ax9.grid(True, alpha=0.3)
    fig9.savefig(FIGURES_DIR / "fig9_pareto_curve.png")
    plt.close(fig9)
    fig_count += 1
    logger.info("  Fig 9 OK -- Pareto Curve")
except Exception as e:
    logger.warning(f"  Fig 9 FAILED: {e}")

# -- Figure 10: Pareto Mmax R2 vs Complexity --
try:
    if all_eq_winner:
        fig10, ax10 = plt.subplots(figsize=(10, 6))
        complexities = [r["complexity"] for r in all_eq_winner]
        mmax_r2s = [r["mmax_R2"] for r in all_eq_winner]
        mmax_mapes = [r["mmax_MAPE"] for r in all_eq_winner]

        ax10_twin = ax10.twinx()
        ax10.plot(complexities, mmax_r2s, "o-", color="#1565C0",
                  markersize=8, linewidth=2, label="Mmax R2")
        ax10_twin.plot(complexities, mmax_mapes, "s--", color="#E65100",
                       markersize=6, linewidth=1.5, alpha=0.7,
                       label="Mmax MAPE %")

        if best_idx_winner is not None and best_idx_winner < len(all_eq_winner):
            best_r = all_eq_winner[best_idx_winner]
            ax10.scatter(best_r["complexity"], best_r["mmax_R2"],
                         s=200, c="red", zorder=5, marker="*",
                         label="Selected")

        ax10.set_xlabel("Equation Complexity")
        ax10.set_ylabel("Mmax R2", color="#1565C0")
        ax10_twin.set_ylabel("Mmax MAPE (%)", color="#E65100")
        ax10.set_title("Pareto Front -- Back-Transformed Performance")
        ax10.legend(loc="lower right", fontsize=10)
        ax10_twin.legend(loc="upper right", fontsize=10)
        ax10.grid(True, alpha=0.3)
        fig10.savefig(FIGURES_DIR / "fig10_pareto_mmax_performance.png")
        plt.close(fig10)
        fig_count += 1
        logger.info("  Fig 10 OK -- Pareto Mmax Performance")
except Exception as e:
    logger.warning(f"  Fig 10 FAILED: {e}")

# -- Figure 11: Ratio vs eta_m (corrosion impact) --
try:
    if WINNER == "RATIO" or True:
        fig11, ax11 = plt.subplots(figsize=(10, 6))
        eta_idx = (safe_names_R.index("eta_m")
                   if "eta_m" in safe_names_R else 0)
        eta_vals = X_ratio[:, eta_idx]
        exp_ratio = y_mmax_R / np.maximum(M_ACI_R, 1e-6)
        pred_ratio = ratio_mmax_pred / np.maximum(M_ACI_R, 1e-6)
        ax11.scatter(eta_vals, exp_ratio, c="#757575", alpha=0.4, s=20,
                     label="Experimental Ratio")
        ax11.scatter(eta_vals, pred_ratio, c="#D32F2F", alpha=0.4, s=20,
                     label="PySR Predicted Ratio")
        ax11.axhline(y=1.0, color="black", linestyle="--", linewidth=1,
                     alpha=0.5, label="Ratio = 1.0 (ACI exact)")
        ax11.set_xlabel("Mass Loss eta_m (%)")
        ax11.set_ylabel("Ratio = Mmax,exp / M_ACI")
        ax11.set_title("Corrosion Correction Factor vs Mass Loss")
        ax11.legend(fontsize=10)
        ax11.grid(True, alpha=0.3)
        fig11.savefig(FIGURES_DIR / "fig11_ratio_vs_eta.png")
        plt.close(fig11)
        fig_count += 1
        logger.info("  Fig 11 OK -- Ratio vs eta_m")
except Exception as e:
    logger.warning(f"  Fig 11 FAILED: {e}")

logger.info(f"  PySR figures generated: {fig_count}")

# =============================================================
# CELL 10: PDF REPORT
# =============================================================
logger.info("\n" + "=" * 60)
logger.info("  Generating PDF Report")
logger.info("=" * 60)


def generate_pdf_report():
    from fpdf import FPDF

    class PDF(FPDF):
        def header(self):
            self.set_font("Helvetica", "B", 10)
            self.set_text_color(13, 27, 42)
            self.cell(
                0, 8,
                "Corrosion RC Beam Optimizer - Scientific Report",
                0, 1, "C",
            )
            self.set_draw_color(189, 189, 189)
            self.line(10, self.get_y(), 200, self.get_y())
            self.ln(3)

        def footer(self):
            self.set_y(-15)
            self.set_font("Helvetica", "I", 8)
            self.set_text_color(128, 128, 128)
            self.cell(0, 10, f"Page {self.page_no()}/{{nb}}", 0, 0, "C")

    pdf = PDF()
    pdf.alias_nb_pages()
    pdf.set_auto_page_break(auto=True, margin=20)

    # -- Title Page --
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 22)
    pdf.ln(40)
    pdf.cell(0, 15, "Corrosion RC Beam Optimizer", 0, 1, "C")
    pdf.set_font("Helvetica", "", 14)
    pdf.cell(0, 10, "Scientific Report - Full Pipeline", 0, 1, "C")
    pdf.set_font("Helvetica", "I", 11)
    pdf.cell(
        0, 10,
        f"Generated: {datetime.now().strftime('%B %d, %Y - %H:%M')}",
        0, 1, "C",
    )
    pdf.ln(10)
    pdf.set_font("Helvetica", "", 10)
    pdf.multi_cell(
        0, 6,
        "This report presents the complete results of the Corrosion RC "
        "Beam Optimizer pipeline. The study applies ML ensemble models and "
        "symbolic regression (PySR) to predict the residual flexural "
        "capacity of corroded RC beams, benchmarked against ACI 318-19.\n"
        f"L1 target: R2 >= {L1_TARGET_R2} | L2 target: R2 >= {L2_TARGET_R2}\n"
        f"Data: {N_TOTAL} specimens | Split: 70/30",
    )

    # -- ACI Benchmark --
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 14)
    pdf.cell(0, 10, "1. ACI 318-19 Benchmark", 0, 1, "L")
    pdf.set_font("Helvetica", "", 10)
    for k, v in aci_metrics.items():
        pdf.cell(0, 7, f"  {k}: {v}", 0, 1)

    # -- Part 1: Ensemble Results --
    pdf.ln(5)
    pdf.set_font("Helvetica", "B", 14)
    pdf.cell(0, 10, "2. Ensemble Model Results (Part 1)", 0, 1, "L")
    pdf.set_font("Helvetica", "", 10)
    if part1_summary:
        bn = part1_summary.get("best_model_name", "?")
        pdf.cell(0, 7, f"  Best Model: {bn}", 0, 1)
        tm = part1_summary.get("test_metrics", {})
        pdf.cell(0, 7, f"  Test R2    = {tm.get('R2', '?')}", 0, 1)
        pdf.cell(0, 7, f"  Test RMSE  = {tm.get('RMSE', '?')} kN.m", 0, 1)
        pdf.cell(0, 7, f"  Test MAE   = {tm.get('MAE', '?')} kN.m", 0, 1)
        pdf.cell(0, 7, f"  Test CV%   = {tm.get('CV_pct', '?')}%", 0, 1)
        pdf.cell(0, 7, f"  Test SD/M  = {tm.get('SD_M', '?')}", 0, 1)
        pdf.ln(3)
        cm = part1_summary.get("cv_all_metrics", {})
        pdf.set_font("Helvetica", "B", 11)
        pdf.cell(0, 7,
                 f"  10-Fold CV (ALL {cm.get('n_samples', '?')} samples):",
                 0, 1)
        pdf.set_font("Helvetica", "", 10)
        pdf.cell(0, 7, f"    R2    = {cm.get('R2', '?')}", 0, 1)
        pdf.cell(0, 7, f"    RMSE  = {cm.get('RMSE', '?')} kN.m", 0, 1)
        pdf.cell(0, 7, f"    MAE   = {cm.get('MAE', '?')} kN.m", 0, 1)
        pdf.cell(0, 7, f"    CV%   = {cm.get('CV_pct', '?')}%", 0, 1)
        pdf.cell(0, 7, f"    SD/M  = {cm.get('SD_M', '?')}", 0, 1)
    else:
        pdf.cell(0, 7, "  (Part 1 results not available)", 0, 1)

    # -- PySR Results --
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 14)
    pdf.cell(0, 10, "3. PySR Symbolic Regression (Part 2)", 0, 1, "L")
    pdf.set_font("Helvetica", "", 10)
    pdf.cell(0, 7, f"  Winner: {WINNER}", 0, 1)
    pdf.cell(0, 7, f"  Equation: {best_eq_str[:90]}", 0, 1)
    pdf.cell(0, 7, f"  Mmax R2   = {r2_mmax:.4f}", 0, 1)
    pdf.cell(0, 7, f"  Mmax RMSE = {rmse_mmax:.4f} kN.m", 0, 1)
    pdf.cell(0, 7, f"  Mmax MAE  = {mae_mmax:.4f} kN.m", 0, 1)
    pdf.cell(0, 7, f"  Mmax MAPE = {mape_mmax:.2f}%", 0, 1)
    pdf.cell(0, 7, f"  CV%       = {cv_pct_eq:.2f}%", 0, 1)
    pdf.cell(0, 7, f"  SD/M      = {sd_m_eq:.4f}", 0, 1)
    pdf.cell(0, 7, f"  L1 broken = {pysr_metrics['L1_broken']}", 0, 1)
    pdf.cell(0, 7, f"  L2 broken = {pysr_metrics['L2_broken']}", 0, 1)
    pdf.ln(3)
    pdf.cell(0, 7, f"  Ratio approach  R2 = {best_R_r2:.4f}", 0, 1)
    pdf.cell(0, 7, f"  Direct approach R2 = {best_D_r2:.4f}", 0, 1)

    # -- Pareto Table --
    pdf.ln(5)
    pdf.set_font("Helvetica", "B", 14)
    pdf.cell(0, 10, "4. Pareto Front Equations", 0, 1, "L")
    pdf.set_font("Helvetica", "", 8)
    for res in all_eq_winner:
        marker = " <<<" if res["index"] == best_idx_winner else ""
        pdf.cell(
            0, 5,
            f"  C={res['complexity']:2d} | Mmax R2={res['mmax_R2']:.4f} | "
            f"MAPE={res['mmax_MAPE']:.1f}%{marker}",
            0, 1,
        )

    # -- Figures Gallery --
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 14)
    pdf.cell(0, 10, "5. Figures Gallery", 0, 1, "L")

    figure_files = sorted(FIGURES_DIR.glob("*.png"))
    for fig_path in figure_files:
        try:
            if pdf.get_y() > 180:
                pdf.add_page()
            pdf.set_font("Helvetica", "I", 9)
            caption = fig_path.stem.replace("_", " ").title()
            pdf.cell(0, 6, caption, 0, 1, "C")
            pdf.image(str(fig_path), x=15, w=180)
            pdf.ln(5)
        except Exception as e:
            pdf.cell(0, 6, f"[Could not embed {fig_path.name}: {e}]", 0, 1)

    report_path = RESULTS_DIR / "Final_Report.pdf"
    pdf.output(str(report_path))
    return report_path


try:
    report_path = generate_pdf_report()
    logger.info(f"PDF Report saved -> {report_path}")
except Exception as e:
    logger.warning(f"PDF report failed: {e}")
    traceback.print_exc()

# =============================================================
# CELL 11: FINAL SUMMARY
# =============================================================
elapsed = time.time() - t_start

sep = "=" * 65
print(f"\n{sep}")
print("  PART 2 COMPLETE -- PySR EQUATION DISCOVERY")
print(sep)

print(f"\n  === PySR DUAL APPROACH ===")
print(f"  Ratio approach  : Mmax R2 = {best_R_r2:.4f}")
print(f"  Direct approach : Mmax R2 = {best_D_r2:.4f}")
print(f"\n  >>> WINNER: {WINNER}")
print(f"  PUBLICATION EQUATION:")
if WINNER == "DIRECT":
    print(f"    Mmax = {best_eq_str}")
else:
    print(f"    Mmax = M_ACI * f_corr")
    print(f"    f_corr = {best_eq_str}")
print(f"\n    R2    = {r2_mmax:.4f}")
print(f"    RMSE  = {rmse_mmax:.4f} kN.m")
print(f"    MAE   = {mae_mmax:.4f} kN.m")
print(f"    MAPE  = {mape_mmax:.2f}%")
print(f"    CV%   = {cv_pct_eq:.2f}%")
print(f"    SD/M  = {sd_m_eq:.4f}")
print(f"    L1    = {pysr_metrics['L1_broken']}")
print(f"    L2    = {pysr_metrics['L2_broken']}")

if part1_summary:
    cm = part1_summary.get("cv_all_metrics", {})
    print(f"\n  === Part 1 Summary (ML) ===")
    print(f"  Best model: {part1_summary.get('best_model_name', '?')}")
    print(f"  10-Fold CV R2 = {cm.get('R2', '?')} "
          f"({cm.get('n_samples', '?')} samples)")

print(f"\n  PDF Report: {RESULTS_DIR / 'Final_Report.pdf'}")
print(f"  Equations:  {EQ_DIR}")
print(f"  PySR time:  {elapsed / 60:.1f} min ({elapsed:.0f}s)")
print(sep)

# =============================================================
# CELL 12: ZIP FOR DOWNLOAD
# =============================================================
import zipfile

zip_path = "/content/final_results_complete.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for sub in ["figures", "models", "equations", "for_part2", "logs"]:
        sub_dir = RESULTS_DIR / sub
        if sub_dir.exists():
            for fpath in sub_dir.rglob("*"):
                if fpath.is_file():
                    arcname = f"{sub}/{fpath.relative_to(sub_dir)}"
                    zf.write(str(fpath), arcname)
    report_f = RESULTS_DIR / "Final_Report.pdf"
    if report_f.exists():
        zf.write(str(report_f), "Final_Report.pdf")

print(f"\nClean ZIP -> {zip_path}")
print("   To download:")
print("   from google.colab import files; "
      "files.download('/content/final_results_complete.zip')")
print("\nDone. Exit code: 0")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 11.5 MB/s eta 0:00:00


13:19:54 | INFO     | =================================================================
13:19:54 | INFO     |   Corrosion RC Beam Optimizer -- Part 2: PySR Equations
13:19:54 | INFO     |   Started: 2026-04-15 13:19:54
13:19:54 | INFO     | =================================================================
13:19:54 | INFO     | Re-running preprocessing + ACI ...
13:19:54 | INFO     | ══════════════════════════════════════════════════
13:19:54 | INFO     |  Starting Preprocessing Pipeline
13:19:54 | INFO     | ══════════════════════════════════════════════════
13:19:54 | INFO     | Loading raw data from: /content/corrosion-rc-beam-optimizer/data/Database.csv
13:19:55 | INFO     | Raw data loaded (encoding=utf-8-sig) — shape: (804, 60)
13:19:55 | INFO     | Column names normalised: ['No.', 'Mass Loss (Tensile bars), ηm (%)', 'Mmax,exp (kNm)', 'Mmax,exp (kNm)']
13:19:55 | INFO     | After column fix — shape: (804, 59)
13:19:55 | INFO     | === Dataset Inspection ===
13:19:55 | INFO     |  

Setup complete.
[juliapkg] Found dependencies: /usr/local/lib/python3.12/dist-packages/pysr/juliapkg.json
[juliapkg] Found dependencies: /usr/local/lib/python3.12/dist-packages/juliapkg/juliapkg.json
[juliapkg] Found dependencies: /usr/local/lib/python3.12/dist-packages/juliacall/juliapkg.json
[juliapkg] Locating Julia 1.10.3 - 1.11
[juliapkg] Using Julia 1.11.5 at /usr/local/bin/julia
[juliapkg] Using Julia project at /root/.julia/environments/pyjuliapkg
[juliapkg] Writing Project.toml:
           | [deps]
           | SymbolicRegression = "8254be44-1295-4e6a-a16d-46603ac705cb"
           | Serialization = "9e88b42a-f829-5b0c-bbe9-9e923198166b"
           | PythonCall = "6099a3de-0909-46bc-b1f4-468b9a2dfc0d"
           | OpenSSL_jll = "458c3c95-2e84-50aa-8efc-19380b2a3a95"
           | 
           | [compat]
           | SymbolicRegression = "~1.11"
           | Serialization = "^1"
           | PythonCall = "=0.9.26"
           | OpenSSL_jll = "~3.0"
[juliapkg] Installing packages:
 

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed pixi_jll ───────────────── v0.41.3+0
   Installed ScientificTypesBase ────── v3.1.0
   Installed MicroMamba ─────────────── v0.1.15
   Installed Tricks ─────────────────── v0.1.13
   Installed Adapt ──────────────────── v4.5.2
   Installed JSON ───────────────────── v1.5.0
   Installed DynamicExpressions ─────── v1.10.4
   Installed PythonCall ─────────────── v0.9.26
   Installed StatisticalTraits ──────── v3.5.0
   Installed PositiveFactorizations ─── v0.2.4
   Installed Optim ──────────────────── v1.13.3
   Installed Pidfile ────────────────── v1.3.0
   Installed ProgressMeter ──────────── v1.10.2
   Installed MLJModelInterface ──────── v1.11.1
   Installed OpenSSL_jll ────────────── v3.0.20+0
   Installed micromamba_jll ─────────── v2.3.1+0
   Installed PtrArrays ──────────────── v1.4.0
   Installed Preferences ────────────── v1.5.2
   Installed DynamicDiff ────────────── v0.2.1

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


...


Expressions evaluated per second: 7.750e+04
Progress: 401 / 24000 total iterations (1.671%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.834e+02  0.000e+00  y = 22.698
2           5.696e+02  2.383e-02  y = sqrt(d)
3           2.894e+02  6.772e-01  y = d * 0.13564
5           2.202e+02  1.366e-01  y = (db_t * 4.802) + -39.692
7           2.154e+02  1.108e-02  y = ((db_t + -8.1101) * 4.8584) - rho_t
8           1.396e+02  4.337e-01  y = ((sqrt(d) + -8.0381) * 9.2306) + -32.344
9           1.121e+02  2.195e-01  y = (((d - eta_m) + -61.513) - 55.074) * 0.31279
11          1.086e+02  1.553e-02  y = (((d + -60.336) - (eta_m + 55.529)) * 0.29738) + rho_t
13          1.075e+02  5.375e-03  y = (((d + -61.162) + (-1.8891 - (eta_m + 56.804))) * 0.31...
                                  

[ Info: Final population:
[ Info: Results saved to:
13:33:39 | INFO     |   Direct training complete.


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.421e+02  0.000e+00  y = db_t
3           2.894e+02  3.138e-01  y = d * 0.13564
4           2.823e+02  2.487e-02  y = exp(db_t ^ 0.45301)
5           1.251e+02  8.134e-01  y = (d * 0.3124) + -39.222
6           8.393e+01  3.993e-01  y = sqrt(RI) * (d + -130.47)
7           8.256e+01  1.650e-02  y = (d + -126.38) * (RI + 0.20027)
8           7.446e+01  1.032e-01  y = sqrt(RI) * ((fc + d) + -165.9)
9           7.287e+01  2.165e-02  y = sqrt(RI) * ((-459.44 / log(fc)) + d)
10          6.608e+01  9.782e-02  y = (d + -101.98) * (RI + exp(-358.35 / b))
12          6.137e+01  3.692e-02  y = ((d - eta_m) + -97.722) * (exp(-350.69 / b) + RI)
13          5.556e+01  9.944e-02  y = ((RI ^ 0.32762) - (31.297 / b)) * (d + (fc + -154.77))
14          5.473e+01  1.508e-02  y = ((d + fc) + -145.64) * (sqrt(RI + 0.09313) - (30.11 / ...
              

13:33:41 | INFO     | ============================================================
13:33:41 | INFO     |   Evaluating Direct Pareto Equations
13:33:41 | INFO     | ============================================================
13:33:41 | INFO     |   D| C= 1 R2=0.0708 RMSE=23.28 MAPE=89.0% | db_t
13:33:41 | INFO     |   D| C= 3 R2=0.5040 RMSE=17.01 MAPE=185.4% | d*0.13564135
13:33:41 | INFO     |   D| C= 4 R2=0.5162 RMSE=16.80 MAPE=165.7% | exp(db_t**0.45301405)
13:33:41 | INFO     |   D| C= 5 R2=0.7893 RMSE=11.09 MAPE=50.3% | d*0.31239852 - 39.22231
13:33:41 | INFO     |   D| C= 6 R2=0.8614 RMSE=8.99 MAPE=39.6% | sqrt(RI)*(d - 130.46901)
13:33:41 | INFO     |   D| C= 7 R2=0.8620 RMSE=8.97 MAPE=42.8% | (RI + 0.20026912)*(d - 126.38454)
13:33:41 | INFO     |   D| C= 8 R2=0.8803 RMSE=8.36 MAPE=40.9% | sqrt(RI)*(d + fc - 165.90439)
13:33:41 | INFO     |   D| C= 9 R2=0.8829 RMSE=8.27 MAPE=42.2% | sqrt(RI)*(d - 459.43823/log(fc))
13:33:41 | INFO     |   D| C=10 R2=0.8868 RMSE=8.13 MAPE=31.1% 


  PART 2 COMPLETE -- PySR EQUATION DISCOVERY

  === PySR DUAL APPROACH ===
  Ratio approach  : Mmax R2 = 0.9228
  Direct approach : Mmax R2 = 0.9322

  >>> WINNER: DIRECT
  PUBLICATION EQUATION:
    Mmax = (-rho_t + log(fy))**rho_t + (RI + 0.4300225 - 44.068012/b)*(-0.15586406*CSI + d + fc - 159.56635)

    R2    = 0.9322
    RMSE  = 6.2901 kN.m
    MAE   = 4.0347 kN.m
    MAPE  = 38.74%
    CV%   = 27.71%
    SD/M  = 0.2770
    L1    = True
    L2    = False

  === Part 1 Summary (ML) ===
  Best model: CatBoost
  10-Fold CV R2 = 0.9752 (804 samples)

  PDF Report: /content/corrosion-rc-beam-optimizer/final_results/Final_Report.pdf
  Equations:  /content/corrosion-rc-beam-optimizer/final_results/equations
  PySR time:  13.9 min (833s)

Clean ZIP -> /content/final_results_complete.zip
   To download:
   from google.colab import files; files.download('/content/final_results_complete.zip')

Done. Exit code: 0
  - outputs/20260415_132852_ngaVl3/hall_of_fame.csv
